In [1]:
import os
import re
import json
from pathlib import Path
from bs4 import BeautifulSoup
import csv

In [2]:
def parse_readvars_csv(csv_path: str | Path) -> dict:
    """Parse EnergyPlus ReadVars CSV (eplusout.csv) into a dict of timeseries.

    Heuristics used:
    - Skip the first header row(s) until the line that starts with "" (blank first cell)
      followed by variable columns (older ReadVars sometimes include metadata rows).
    - Detect whether the CSV is hourly by counting rows (>= 8000 rows is treated as yearly hourly).
    - Return a dict mapping sanitized variable names to numeric arrays.

    This implementation is conservative: it reads numeric columns only and limits
    memory by only reading the first ~20000 rows.
    """
    csv_path = Path(csv_path)
    series = {}
    try:
        with csv_path.open('r', encoding='utf-8', errors='replace') as fh:
            reader = csv.reader(fh)
            rows = []
            # Read all rows up to a reasonable cap (protect memory)
            max_rows = 20000
            for i, row in enumerate(reader):
                rows.append(row)
                if i + 1 >= max_rows:
                    break

        if len(rows) < 2:
            return {}

        # Attempt to find the header row which contains variable names
        header_idx = 0
        # Common EnergyPlus ReadVars CSV has first column blank then variable labels
        for idx, r in enumerate(rows[:10]):
            # Heuristic: header row contains at least 2 non-empty cells and not numeric
            non_empty = sum(1 for c in r if c and c.strip() != '')
            if non_empty >= 2 and any(not _is_number_like(c) for c in r):
                header_idx = idx
                break

        header = rows[header_idx]
        data_rows = rows[header_idx+1:]

        # If the first column is empty and second column looks like a number for the
        # first data row, then columns from 1..N are variables
        col_count = len(header)
        if col_count < 2:
            return {}

        # Determine if this looks like an hourly file (approx 8760 rows)
        hourly_like = len(data_rows) >= 8000

        # For each column, try to parse numeric values
        for col_idx in range(1, col_count):
            var_name_raw = header[col_idx].strip() if header[col_idx] else f'col_{col_idx}'
            var_name = _sanitize_variable_name(var_name_raw)
            vals = []
            for r in data_rows:
                if col_idx >= len(r):
                    vals.append(None)
                    continue
                v = r[col_idx].strip()
                if v == '' or v == '\u00a0':
                    vals.append(None)
                    continue
                try:
                    vals.append(float(v))
                except Exception:
                    vals.append(None)
            # If hourly_like but we have far fewer rows, skip this variable
            if hourly_like and len([x for x in vals if x is not None]) < 100:
                continue
            series[var_name] = vals

        # If nothing meaningful found, return empty
        if not series:
            return {}

        # Attach metadata
        return {
            'is_hourly': hourly_like,
            'rows': len(data_rows),
            'series': series
        }
    except Exception:
        return {}


def _sanitize_variable_name(name: str) -> str:
    """Remove units in parentheses and replace spaces with underscores"""
    s = re.sub(r"\(.*?\)", '', name).strip()
    s = re.sub(r"[^0-9A-Za-z_]+", '_', s)
    s = s.strip('_')
    return s or name


def _is_number_like(s: str) -> bool:
    """Check if a string can be converted to a float"""
    try:
        float(s)
        return True
    except Exception:
        return False

In [ ]:
def parse_html_with_table_lookup(html_path, log_path, file_name=None):
    """Parse EnergyPlus HTML output and log to extract structured results.
    
    Args:
        html_path: Path to the EnergyPlus HTML output file (typically output.htm or *Table.html)
        log_path: Path to the run log file (typically run_output.log) - can be None or missing
        file_name: Optional original IDF filename for reference
        
    Returns:
        Dictionary with parsed results including energy use, zones, and runtime
    """
    try:
        with open(html_path, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f, 'html.parser')
        
        # Try to read log file, but don't fail if it doesn't exist
        log_content = ""
        if log_path and Path(log_path).exists():
            with open(log_path, 'r', encoding='utf-8') as f:
                log_content = f.read()

        def get_value_from_table(table_title, row_label, col_index=1):
            """Locate a table by title and return value at row_label and column index."""
            table_header = soup.find('b', string=table_title)
            if not table_header:
                return None
            table = table_header.find_next('table')
            if not table:
                return None
            for row in table.find_all('tr'):
                cells = row.find_all('td')
                if cells and row_label in cells[0].text.strip():
                    try:
                        return float(cells[col_index].text.strip())
                    except ValueError:
                        return None
            return None

        # Extract building name
        building_name = soup.find('p', string=lambda s: s and 'Building:' in s)
        building_name = building_name.b.text if building_name and building_name.b else "Unknown Building"

        # Use provided file_name or fallback
        if not file_name:
            file_name = "unknown.idf"

        # Extract values from tables
        total_energy_use = get_value_from_table("Site and Source Energy", "Total Site Energy", 2)  # kWh/m²
        total_area = get_value_from_table("Building Area", "Total Building Area")  # m²
        
        # Extract energy values from End Uses table
        # "Heating" = space heating, "Water Systems" = domestic hot water
        heating_kwh = get_value_from_table("End Uses", "Heating", 12)  # District Heating column
        water_systems_kwh = get_value_from_table("End Uses", "Water Systems", 12)  # DHW
        cooling_kwh = get_value_from_table("End Uses", "Cooling", 12)
        lighting_kwh = get_value_from_table("End Uses", "Interior Lighting", 1)
        equipment_kwh = get_value_from_table("End Uses", "Interior Equipment", 1)

        if not total_area:
            total_area = 1.0  # fallback to avoid division by zero

        # Normalize values by area
        heating_demand = heating_kwh / total_area if heating_kwh else 0.0  # Space heating only
        dhw_demand = water_systems_kwh / total_area if water_systems_kwh else 0.0  # DHW only
        total_heating_demand = heating_demand + dhw_demand  # Combined heating
        cooling_demand = cooling_kwh / total_area if cooling_kwh else 0.0
        lighting_intensity = lighting_kwh / total_area if lighting_kwh else 0.0
        equipment_intensity = equipment_kwh / total_area if equipment_kwh else 0.0

        # Extract run time in seconds from log (if available)
        runtime_seconds = 0.0
        if log_content:
            runtime_match = re.search(r'EnergyPlus Run Time=(\d+)hr\s+(\d+)min\s+([\d\.]+)sec', log_content)
            if runtime_match:
                hr, minute, sec = map(float, runtime_match.groups())
                runtime_seconds = hr * 3600 + minute * 60 + sec

        # Extract energy use breakdown by end use
        energy_use = {}
        table_header = soup.find('b', string="End Uses")
        if table_header:
            table = table_header.find_next('table')
            if table:
                for row in table.find_all('tr'):
                    cells = row.find_all('td')
                    if len(cells) > 1:
                        end_use = cells[0].text.strip()
                        if end_use and end_use not in ["", "&nbsp;", "Total End Uses"]:
                            # Get values for electricity and district heating
                            electricity = cells[1].text.strip() if len(cells) > 1 else "0"
                            district_heating = cells[12].text.strip() if len(cells) > 12 else "0"
                            try:
                                electricity = float(electricity) if electricity else 0.0
                                district_heating = float(district_heating) if district_heating else 0.0
                                energy_use[end_use] = {
                                    "electricity": electricity,
                                    "district_heating": district_heating,
                                    "total": electricity + district_heating
                                }
                            except ValueError:
                                pass

        # Extract zone information
        # Try both "Zone Summary" (older format) and "Zone Information" (newer format)
        zones = []
        zone_table_header = soup.find('b', string="Zone Summary")
        if not zone_table_header:
            zone_table_header = soup.find('b', string="Zone Information")
        
        if zone_table_header:
            print(f"Found zone table with header: {zone_table_header.text}")
            table = zone_table_header.find_next('table')
            if table:
                rows_processed = 0
                for row in table.find_all('tr'):
                    cells = row.find_all('td')
                    # Skip header row and total rows
                    if len(cells) < 4:
                        continue
                    
                    rows_processed += 1
                    
                    # Check if this is a data row (first cell is usually a number or zone name)
                    first_cell = cells[0].text.strip()
                    if not first_cell or "Total" in first_cell:
                        continue
                    
                    try:
                        # For "Zone Summary" format: zone name in col 0, area in col 1, volume in col 4
                        # For "Zone Information" format: index in col 0, name in col 1, area in col 22, volume in col 19
                        if len(cells) > 22:  # Zone Information format
                            zone_name = cells[1].text.strip()
                            area = float(cells[22].text.strip())
                            volume = float(cells[19].text.strip())
                        else:  # Zone Summary format
                            zone_name = cells[0].text.strip()
                            area = float(cells[1].text.strip())
                            volume = float(cells[4].text.strip())
                        
                        zones.append({
                            "name": zone_name,
                            "area": area,
                            "volume": volume
                        })
                    except (ValueError, IndexError) as e:
                        # Skip rows that can't be parsed
                        print(f"Skipped row with {len(cells)} cells, first cell: {first_cell[:50] if first_cell else 'empty'}, error: {e}")
                        pass
                
                print(f"Processed {rows_processed} table rows, found {len(zones)} zones")
            else:
                print("Zone table header found but no table element after it")
        else:
            print("No zone table header found ('Zone Summary' or 'Zone Information')")

        # Combine results into a structured output
        result = {
            "building": building_name,
            "fileName": file_name,
            "totalEnergyUse": round(total_energy_use, 1) if total_energy_use else 0.0,
            "spaceHeatingDemand": round(heating_demand, 1),  # Space heating only
            "dhwDemand": round(dhw_demand, 1),  # Domestic hot water only
            "totalHeatingDemand": round(total_heating_demand, 1),  # Space + DHW
            "heatingDemand": round(heating_demand, 1),  # Keep for backwards compatibility
            "coolingDemand": round(cooling_demand, 1),
            "lightingDemand": round(lighting_intensity, 1),
            "equipmentDemand": round(equipment_intensity, 1),
            "runTime": round(runtime_seconds, 1),
            "totalArea": total_area,
            "energy_use": energy_use,
            "zones": zones,
            "status": "success"
        }

        return result

    except Exception as e:
        import traceback
        traceback.print_exc()
        
        return {
            "error": str(e),
            "fileName": file_name or "unknown.idf",
            "totalEnergyUse": 0.0,
            "heatingDemand": 0.0,
            "coolingDemand": 0.0,
            "runTime": 0.0,
            "energy_use": {},
            "zones": [],
            "status": "error"
        }

In [4]:
def process_simulation_results(output_dir, file_name=None):
    """Process EnergyPlus simulation results from a directory.
    
    Args:
        output_dir: Directory containing the simulation output files
        file_name: Optional original IDF filename for reference (e.g., "city_optimized.idf")
        
    Returns:
        Dictionary with parsed results including hourly timeseries if available
    """
    output_dir = Path(output_dir)
    
    # Try multiple naming patterns for the HTML and log files
    # Pattern 1: Standard EnergyPlus output naming
    html_path = output_dir / 'output.htm'
    log_path = output_dir / 'run_output.log'
    csv_path = output_dir / 'output.csv'
    
    # Pattern 2: If file_name is provided, derive the pattern (e.g., city_optimizedTable.html)
    if file_name and not html_path.exists():
        base_name = Path(file_name).stem  # Remove .idf extension
        html_path = output_dir / f'{base_name}Table.html'
        csv_path = output_dir / f'{base_name}.csv'
    
    # Pattern 3: Check for eplustbl.htm (original EnergyPlus name)
    if not html_path.exists():
        html_path = output_dir / 'eplustbl.htm'
    
    print(f"Looking for HTML at: {html_path}")
    print(f"Looking for log at: {log_path}")
    print(f"HTML exists: {html_path.exists()}")
    print(f"Log exists: {log_path.exists()}")
    
    if html_path.exists():
        # Extract results from HTML
        results = parse_html_with_table_lookup(html_path, log_path, file_name)

        # If a ReadVars CSV exists, attempt to parse hourly timeseries
        if csv_path.exists():
            try:
                print(f"Found CSV file at: {csv_path}")
                hourly_data = parse_readvars_csv(csv_path)
                if hourly_data:
                    results['hourly_timeseries'] = hourly_data
                    print(f"Parsed {len(hourly_data.get('series', {}))} timeseries variables")
            except Exception as e:
                print(f"Error parsing CSV: {e}")
                # Non-fatal: continue even if CSV parsing fails
                pass
        
        # Add original filename to results if provided
        if file_name:
            results['originalFileName'] = file_name
        
        # Save parsed results as a JSON file
        json_path = output_dir / 'parsed_results.json'
        with open(json_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Saved parsed results to: {json_path}")
        
        return results
    else:
        print(f"Warning: HTML results file not found at {html_path}")
        # List available files to help debug
        try:
            available_files = [f.name for f in output_dir.iterdir() if f.is_file()]
            print(f"Available files in directory: {available_files[:10]}")  # Show first 10
        except Exception:
            pass
        
        return {
            'error': 'HTML results file not found',
            'fileName': file_name or "unknown.idf",
            'originalFileName': file_name,
            'status': 'error'
        }

## Example Usage

Now you can use these functions to parse EnergyPlus simulation results without Django dependencies.

In [5]:
# Example 1: Parse a single simulation result directory
# Using the actual output directory in notebooks
output_directory = Path(r"idf_output")

# For the optimized city building
results = process_simulation_results(output_directory, file_name="city_optimized.idf")

# View the parsed results (first few keys)
if results['status'] == 'success':
    print(f"Successfully parsed: {results['fileName']}")
    print(f"Building: {results['building']}")
    print(f"Total Energy Use: {results['totalEnergyUse']} kWh/m²")
else:
    print(f"Status: {results['status']}")
    print(f"Error: {results.get('error', 'Unknown error')}")

Looking for HTML at: idf_output\city_optimizedTable.html
Looking for log at: idf_output\run_output.log
HTML exists: True
Log exists: False
Found zone table with header: Zone Summary
Processed 147 table rows, found 142 zones
Found CSV file at: idf_output\city_optimized.csv
Parsed 255 timeseries variables
Saved parsed results to: idf_output\parsed_results.json
Successfully parsed: city_optimized.idf
Building: Unknown Building
Total Energy Use: 276.9 kWh/m²


In [ ]:
# Example 2: Access specific results with separate space heating and DHW
if results['status'] == 'success':
    print(f"Building: {results['building']}")
    print(f"Total Energy Use: {results['totalEnergyUse']} kWh/m²")
    print(f"\n=== Heating Breakdown ===")
    print(f"Space Heating Demand: {results['spaceHeatingDemand']} kWh/m²")
    print(f"DHW Demand: {results['dhwDemand']} kWh/m²")
    print(f"Total Heating Demand: {results['totalHeatingDemand']} kWh/m²")
    print(f"\n=== Other ===")
    print(f"Cooling Demand: {results['coolingDemand']} kWh/m²")
    print(f"Total Area: {results['totalArea']} m²")
    print(f"Runtime: {results['runTime']} seconds")
    print(f"\nNumber of zones: {len(results['zones'])}")
    
    # Check if hourly data is available
    if 'hourly_timeseries' in results:
        hourly = results['hourly_timeseries']
        print(f"\nHourly data available: {hourly['is_hourly']}")
        print(f"Number of rows: {hourly['rows']}")
        print(f"Available variables: {list(hourly['series'].keys())[:10]}")  # First 10 variables
else:
    print(f"Error: {results.get('error', 'Unknown error')}")

Building: Unknown Building
Total Energy Use: 276.9 kWh/m²
Heating Demand: 192.7 kWh/m²
Cooling Demand: 0.0 kWh/m²
Total Area: 313443.75 m²
Runtime: 0.0 seconds

Number of zones: 142

Hourly data available: True
Number of rows: 8760
Available variables: ['BUILDING0_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING1_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING2_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING3_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING4_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING5_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING6_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING7_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING8_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING9_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J']


In [7]:
# Example 3: Parse just the HTML file directly
# Using the actual HTML file from idf_output
html_file = Path(r"idf_output/city_optimizedTable.html")
log_file = Path(r"idf_output/run_output.log")  # May not exist

if html_file.exists():
    results_direct = parse_html_with_table_lookup(html_file, log_file, "city_optimized.idf")
    print(f"Building: {results_direct['building']}")
    print(f"Total Energy: {results_direct['totalEnergyUse']} kWh/m²")
    print(f"Zones found: {len(results_direct['zones'])}")
    if results_direct['zones']:
        print(f"First zone: {results_direct['zones'][0]['name']} - {results_direct['zones'][0]['area']} m²")
else:
    print(f"HTML file not found: {html_file}")

Found zone table with header: Zone Summary
Processed 147 table rows, found 142 zones
Building: Unknown Building
Total Energy: 276.9 kWh/m²
Zones found: 142
First zone: BUILDING0_FLOOR1_ROOM1 - 655.3 m²


In [8]:
# Example 4: Parse hourly timeseries CSV
# Using the actual CSV file from idf_output
csv_file = Path(r"idf_output/city_optimized.csv")

if csv_file.exists():
    hourly_data = parse_readvars_csv(csv_file)
    
    if hourly_data:
        print(f"Is hourly: {hourly_data['is_hourly']}")
        print(f"Number of rows: {hourly_data['rows']}")
        print(f"Number of variables: {len(hourly_data['series'])}")
        
        # Show first 3 variables as examples
        print(f"\nFirst 3 variables:")
        for var_name in list(hourly_data['series'].keys())[:3]:
            values = hourly_data['series'][var_name]
            print(f"  {var_name}: {len([v for v in values if v is not None])} non-null values")
    else:
        print("No hourly data parsed from CSV")
else:
    print(f"CSV file not found: {csv_file}")

Is hourly: True
Number of rows: 8760
Number of variables: 255

First 3 variables:
  BUILDING0_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J: 8760 non-null values
  BUILDING1_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J: 8760 non-null values
  BUILDING2_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J: 8760 non-null values


Parse results